# Noise Measurement

## Instantiation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep
from scipy import signal
from scipy.signal import decimate
from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qstl_instruments.qstl_nidaq import QSTL_NIDaq
%matplotlib inline

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)
t_acqu = 10 # acquisition time in second
decimation = 500
meas_seg = 6
total_meas_time = 3600 * 10
t = np.arange(start=0, stop=t_acqu, step=1/daq.max_sampling_rate)
station = Station()

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251117_NoiseData/noise3.db")
exp = load_or_create_experiment("2D sweep", "4MegaOhmTermination_noise_data")
meas = Measurement(exp=exp, station=station)

meas_time = Parameter(name = "meas_time", label = "Measurement Time", unit = "s")
meas_freq = Parameter(name = "meas_freq", label = "Measurement Frequency", unit = "Hz")
meas_rms = Parameter(name = "meas_rms", label = "Measurement RMS", unit = "pA")
trace_data = Parameter(name = "trace_data", label = "Measurement Data", unit = "pA")
fft_data = Parameter(name = "fft_data", label = "FFT Data", unit = "A/Hz^-1/2")
elapsed_time = Parameter(name = "elapsed_time", label = "Elapsed Time", unit = "s")
meas.register_parameter(meas_time)
meas.register_parameter(meas_freq)
meas.register_parameter(elapsed_time)
meas.register_parameter(fft_data, setpoints=(meas_freq, elapsed_time))
meas.register_parameter(trace_data, setpoints=(meas_time, elapsed_time))
meas.register_parameter(meas_rms, setpoints=(elapsed_time,))

## Measurement

In [2]:
start_time = time()

with meas.run() as datasaver:
    while True:
        num_of_samples = int(daq.max_sampling_rate * t_acqu)
        voltage_i = daq.read(
            ch = "Dev2/ai1",
            num_of_samples = num_of_samples
        )
        voltage_i = np.array(voltage_i[10:])
        current_i = daq.convert_volts_to_amps(voltage_i)

        i_desired = decimate(current_i, decimation, ftype="fir", zero_phase=True)
        t_desired = np.linspace(0, t_acqu, len(i_desired))

        rms_pA = 1e12 * np.sqrt(np.mean((i_desired - np.mean((i_desired)))**2))
        print(f"RMS noise {rms_pA} pA")

        f, II_den = signal.periodogram(
            i_desired,
            fs = daq.max_sampling_rate/decimation,
            window = "flattop",
            scaling = "density",
            return_onesided = True
        )

        current_time = time() - start_time

        datasaver.add_result(
            (elapsed_time, [current_time] * len(f)),
            (meas_freq, f),
            (fft_data, np.sqrt(II_den))
        )
        datasaver.add_result(
            (elapsed_time, [current_time] * len(t_desired)),
            (meas_time, t_desired),
            (trace_data, 1e12 * i_desired)
        )
        datasaver.add_result(
            (elapsed_time, current_time),
            (meas_rms, rms_pA)
        )
        sleep((meas_seg - 1) * t_acqu)
        if current_time > total_meas_time:
            break

Starting experimental run with id: 13. 
RMS noise 2.0837908524301656 pA
RMS noise 2.073991725267569 pA
RMS noise 2.07510600341157 pA
RMS noise 2.0795711992044277 pA
RMS noise 2.0931005643200837 pA
RMS noise 2.0817816604687174 pA
RMS noise 2.0879976108987828 pA
RMS noise 2.1061726016526072 pA
RMS noise 2.079874520576786 pA
RMS noise 2.1031141767857346 pA
RMS noise 2.0832177426423275 pA
RMS noise 2.0980998341797603 pA
RMS noise 2.1028111944628036 pA
RMS noise 2.105801099709815 pA
RMS noise 2.0901812028374356 pA
RMS noise 2.0928591808594263 pA
RMS noise 2.096369030445231 pA
RMS noise 2.1109420291635645 pA
RMS noise 2.098539751776267 pA
RMS noise 2.104325218978415 pA
RMS noise 2.086778182541474 pA
RMS noise 2.100040741381209 pA
RMS noise 2.0800954643937257 pA
RMS noise 2.082056921383701 pA
RMS noise 2.095953192079864 pA
RMS noise 2.0838724369818387 pA
RMS noise 2.0820830756884647 pA
RMS noise 2.0935165314475963 pA
RMS noise 2.0823723245398704 pA
RMS noise 2.086401103195406 pA
RMS noise 2.0